# Stage 1 - Non-Instruction Fine-Tuning (Domain Adaptation)

**Base model:** Llama-3.2-3B-Instruct (Unsloth, 4-bit / QLoRA)

We first adapt the model to customer-support **domain language** using raw,
free-text paragraphs (continued-pretraining style: plain next-token prediction,
no chat structure yet).

**Input:** `data/non_instruction_data.csv` (column `text`)
**Output:** LoRA adapter -> `models/non_instruct_adapter/`

In [ ]:
%%capture
# Clean, version-consistent install. Let Unsloth pin its own compatible
# transformers/trl/peft versions - do NOT force-upgrade trl separately.
!pip install --upgrade unsloth unsloth_zoo
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Get the project code + datasets into Colab.
# Replace <your-username> with your GitHub username.
!git clone https://github.com/<your-username>/customer-support-assistant.git
%cd customer-support-assistant

import os
DATA = "data"
MODELS = "models"
os.makedirs(MODELS, exist_ok=True)
print("cwd:", os.getcwd())

## 1. Load Llama-3.2-3B-Instruct + attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"   # Llama-3.2-3B-Instruct, 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # auto: fp16 on T4, bf16 on Ampere+
    load_in_4bit = True,     # QLoRA
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # if loss ever stays flat, try = True
    random_state = 3407,
)

## 2. Load raw domain text and add EOS

In [ ]:
from datasets import load_dataset

raw = load_dataset("csv", data_files="data/non_instruction_data.csv", split="train")
print(">>> paragraphs:", len(raw))
print(raw[0]["text"][:300])

EOS = tokenizer.eos_token
raw = raw.map(lambda b: {"text": [t + EOS for t in b["text"]]}, batched=True)

## 3. Train (next-token prediction over domain text)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = raw,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        packing = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage1",
        report_to = "none",
    ),
)
stats = trainer.train()
print("final (mean) loss:", stats.training_loss)   # should fall well below 2.0

## 4. Save the domain-adapted adapter

In [ ]:
model.save_pretrained("models/non_instruct_adapter")
tokenizer.save_pretrained("models/non_instruct_adapter")
print("Saved -> models/non_instruct_adapter")

### Stage 1 done
Optional warm-start for Stage 2. Since Llama-3.2-3B-Instruct is already a strong
instruct model, you can also skip straight to Stage 2 on the base model - both are
acceptable. Next: `instruction_finetuning.ipynb`.